# Module 7.5: HDR Tone Mapping Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_07_RL_For_Image_Enhancement/05_HDR_Tone_Mapping_Agent/hdr_tone_mapping_agent.ipynb)

---

## Learning Objectives

1. **Understand** High Dynamic Range (HDR) imaging and the tone mapping problem
2. **Formulate** tone mapping operator selection as an MDP
3. **Implement** global and local tone mapping operators with tunable parameters
4. **Train** a DQN agent that learns to compress dynamic range while preserving visual quality
5. **Evaluate** using perceptual quality metrics (TMQI) and visual comparison

---

## 1 — Mathematical Foundation

### 1.1 HDR Imaging

Real-world luminance spans many orders of magnitude ($10^{-3}$ to $10^5$ cd/m²), while standard displays reproduce only $\sim$2–3 orders. HDR images store the full range in floating-point, and **tone mapping** compresses this to a displayable Low Dynamic Range (LDR) image.

### 1.2 Global Tone Mapping Operators (TMOs)

**Reinhard global operator:**

$$L_d(x,y) = \frac{L(x,y) / \bar{L}}{1 + L(x,y) / \bar{L}}$$

where $L$ is the luminance, $\bar{L} = \exp\!\left(\frac{1}{N}\sum \log(L + \epsilon)\right)$ is the log-average luminance (key value).

**Reinhard with white point:**

$$L_d = \frac{L(1 + L/L_w^2)}{1 + L}$$

where $L_w$ is the smallest luminance mapped to pure white.

**Logarithmic mapping:**

$$L_d = \frac{\log(1 + \mu \cdot L/L_{\max})}{\log(1 + \mu)}$$

Parameter $\mu$ controls compression strength.

**Gamma compression:**

$$L_d = L^{1/\gamma}$$

### 1.3 Local Tone Mapping

**Reinhard local operator** adapts to local image content using a local adaptation luminance:

$$L_d(x,y) = \frac{L(x,y)}{1 + V_1(x,y,s^*)}$$

where $V_1$ is a Gaussian-filtered version at scale $s^*$ chosen to maximise local contrast without halos.

### 1.4 MDP Formulation

| Component | Definition |
|-----------|------------|
| **State** $s_t$ | HDR luminance histogram features + current LDR image statistics |
| **Action** $a_t$ | Select TMO + parameter setting from discrete set |
| **Transition** | Apply selected TMO to HDR image |
| **Reward** | Perceptual quality: $r_t = \Delta\text{TMQI}$ |

### 1.5 Tone Mapped Image Quality Index (TMQI)

TMQI combines structural fidelity $S$ and statistical naturalness $N$:

$$\text{TMQI} = a \cdot S^\alpha + (1-a) \cdot N^\beta$$

**Structural fidelity** measures how well the TMO preserves local contrast:

$$S = \frac{2\sigma_{xy} + C}{\sigma_x^2 + \sigma_y^2 + C}$$

**Statistical naturalness** measures how close the LDR histogram is to the distribution of natural images:

$$N = \frac{1}{K} \exp\!\left(-\frac{(\mu - \mu_n)^2}{2\sigma_n^2}\right) \cdot \exp\!\left(-\frac{(\sigma - \sigma_n)^2}{2\sigma_{\sigma_n}^2}\right)$$

where $\mu_n \approx 115$, $\sigma_n \approx 44$ are statistics of natural LDR images.

## 2 — Environment Setup

## Deep Derivation: HDR Tone Mapping Operators and Dynamic Range Compression

### Step 1: Dynamic Range — Definition and Scale
**Scene dynamic range:** $\text{DR} = \frac{L_{\max}}{L_{\min}}$ (ratio of max to min luminance)

Real scenes: DR up to $10^6:1$ ($\approx 120$ dB). LDR display: $\approx 200:1$ ($\approx 46$ dB).

**Problem:** Compress $\sim 120$ dB to $\sim 46$ dB while preserving visual detail.

$$\text{Compression ratio} = \frac{\log_2(10^6)}{\log_2(200)} \approx \frac{20}{7.6} \approx 2.6\times \quad \text{(non-trivial!)}$$

### Step 2: Global Tone Mapping — Reinhard Operator
$$L_d(x,y) = \frac{L(x,y)}{1 + L(x,y)}$$

**Proof this maps $[0, \infty) \to [0, 1)$:**
- $L = 0 \implies L_d = 0$ (dark stays dark)
- $L \to \infty \implies L_d \to 1$ (bright saturates)
- $\frac{dL_d}{dL} = \frac{1}{(1+L)^2} > 0$ (monotonically increasing)

Therefore $L_d \in [0, 1)$ for all $L \geq 0$. $\blacksquare$

**Extended Reinhard** (allows burn-out for highlights):
$$L_d = \frac{L(1 + L/L_{\text{white}}^2)}{1 + L}$$

### Step 3: Logarithmic Tone Mapping — Weber-Fechner Law
Human brightness perception is approximately logarithmic:
$$\text{perceived brightness} \propto \log(L)$$

**Drago's logarithmic mapping:**
$$L_d = \frac{\log(1 + L)}{\log(1 + L_{\max})}$$

**Proof this preserves contrast ratios logarithmically:**

For two luminances $L_1, L_2$ with $L_1 > L_2$:
$$\frac{L_{d,1} - L_{d,2}}{L_{d,2}} = \frac{\log(1+L_1) - \log(1+L_2)}{\log(1+L_2)} = \frac{\log\frac{1+L_1}{1+L_2}}{\log(1+L_2)} \quad \blacksquare$$

### Step 4: Local Tone Mapping — Adaptive Key Value
**Key of the scene** (log-average luminance):
$$\bar{L} = \exp\left(\frac{1}{N}\sum_{i=1}^N \log(\delta + L_i)\right)$$

where $\delta$ is small constant to avoid $\log(0)$.

**Proof key value is the geometric mean:**
$$\bar{L} = \exp\left(\frac{1}{N}\sum_i \log L_i\right) = \left(\prod_i L_i\right)^{1/N} = \text{GM}(L_1, \ldots, L_N) \quad \blacksquare$$

**Scaled luminance:** $L_s(x,y) = \frac{a}{\bar{L}} L(x,y)$ where $a \approx 0.18$ (middle gray).

### Step 5: Bilateral Tone Mapping (Durand & Dorsey)
Decompose log-luminance into base and detail layers:
$$\log L = \underbrace{B}_{\text{base (low freq)}} + \underbrace{D}_{\text{detail (high freq)}}$$

where $B = \text{BilateralFilter}(\log L)$ and $D = \log L - B$.

Compress only the base, preserve detail:
$$\log L_{\text{out}} = \alpha \cdot B + D, \quad \alpha < 1$$

**Proof this preserves local contrast:**

Local contrast $\approx D = \log L - B$ is unchanged. Only global range $B$ is compressed by factor $\alpha$. $\blacksquare$

### Step 6: Tone Mapping Quality — TMQI Metric
$$\text{TMQI} = a \cdot S^{\alpha} + (1-a) \cdot N^{\beta}$$

where $S$ = structural fidelity, $N$ = statistical naturalness.

**Statistical naturalness** models the histogram of LDR images as Gaussian in log-domain:
$$N = \frac{1}{\sigma_n\sqrt{2\pi}}\exp\left(-\frac{(\mu - \mu_n)^2}{2\sigma_n^2}\right)$$

### Step 7: RL Agent for Adaptive Tone Mapping
**State:** HDR image statistics (histogram, key value, local contrast map)

**Actions:** adjust tone curve parameters: $a_t \in \{(\Delta\gamma, \Delta\text{key}, \Delta\text{compression}, \Delta\text{detail})\}$

**Reward:** $r_t = w_1 \cdot \Delta\text{TMQI} + w_2 \cdot \Delta\text{detail\_preservation} + w_3 \cdot \Delta\text{naturalness}$

**Connection to RL:** Different HDR scenes (sunset, indoor, night) require different tone curves. An RL agent learns a POLICY that adapts its tone mapping strategy to each image's content — outperforming fixed operators that use the same parameters for all images.

In [ ]:
!pip install torch torchvision numpy opencv-python-headless matplotlib scikit-image gymnasium -q

In [ ]:
# Download REAL open-source datasets for image enhancement
import torchvision
import numpy as np
import urllib.request
import os

# CIFAR-10 real photographs (our base images to enhance)
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
real_images = np.array([np.array(cifar10[i][0]) for i in range(1000)])
print(f"✅ CIFAR-10: {len(real_images)} real photos loaded as ground truth")

# BSD68 denoising benchmark (68 real test images)
bsd_url = "https://raw.githubusercontent.com/clausmichele/CBSD68-dataset/master/CBSD68/original/"
os.makedirs('./data/bsd68', exist_ok=True)
# Download first 10 for demo
for i in range(1, 11):
    fname = f"./data/bsd68/{i:04d}.png"
    if not os.path.exists(fname):
        try:
            urllib.request.urlretrieve(f"{bsd_url}{i:04d}.png", fname)
        except:
            pass
print("✅ BSD68 benchmark: downloading real denoising test images")
print("   These are REAL photographs used in denoising research papers!")

In [ ]:
import os
import random
import math
import collections

import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from skimage.metrics import structural_similarity as compute_ssim
import gymnasium as gym
from gymnasium import spaces

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 3 — Synthetic HDR Dataset

We generate synthetic HDR images with high dynamic range (luminance values spanning several orders of magnitude) and pair them with reference tone-mapped outputs.

In [ ]:
IMG_SIZE = 64
NUM_IMAGES = 60


def generate_hdr_image(size: int = IMG_SIZE) -> np.ndarray:
    """Generate a synthetic HDR image with high dynamic range."""
    img = np.zeros((size, size, 3), dtype=np.float32)

    # Dark background region
    bg_val = random.uniform(0.001, 0.05)
    img[:, :] = bg_val

    # Medium-luminance regions
    for _ in range(random.randint(3, 6)):
        colour = np.array([random.uniform(0.1, 2.0) for _ in range(3)], dtype=np.float32)
        x, y = random.randint(0, size-5), random.randint(0, size-5)
        w, h = random.randint(5, size//3), random.randint(5, size//3)
        img[y:y+h, x:x+w] = colour

    # Bright highlight regions (simulating light sources)
    for _ in range(random.randint(1, 3)):
        cx, cy = random.randint(5, size-5), random.randint(5, size-5)
        r = random.randint(3, size//6)
        intensity = random.uniform(5.0, 50.0)
        yy, xx = np.ogrid[:size, :size]
        dist = np.sqrt((xx - cx)**2 + (yy - cy)**2).astype(np.float32)
        falloff = np.clip(1.0 - dist / (r * 2), 0, 1)
        highlight_colour = np.array([random.uniform(0.8, 1.0) for _ in range(3)], dtype=np.float32)
        img += intensity * falloff[:, :, np.newaxis] * highlight_colour

    # Add smooth gradients
    grad = np.linspace(0.01, random.uniform(1.0, 5.0), size).reshape(1, -1, 1).astype(np.float32)
    img += grad * 0.3

    return np.maximum(img, 1e-6)  # Ensure positive


def reference_tone_map(hdr: np.ndarray) -> np.ndarray:
    """High-quality reference tone mapping using Reinhard global with tuned params."""
    lum = 0.2126 * hdr[:,:,0] + 0.7152 * hdr[:,:,1] + 0.0722 * hdr[:,:,2]
    log_avg = np.exp(np.mean(np.log(lum + 1e-6)))
    key = 0.18
    L_scaled = key * lum / log_avg
    L_white = np.percentile(L_scaled, 99)
    L_d = L_scaled * (1 + L_scaled / (L_white**2 + 1e-6)) / (1 + L_scaled)
    scale = L_d / (lum + 1e-6)
    ldr = hdr * scale[:, :, np.newaxis]
    # Gamma correction
    ldr = np.power(np.clip(ldr, 0, 1), 1/2.2)
    return np.clip(ldr, 0, 1).astype(np.float32)


dataset = []
for _ in range(NUM_IMAGES):
    hdr = generate_hdr_image()
    ref_ldr = reference_tone_map(hdr)
    dataset.append({"hdr": hdr, "reference": ref_ldr})

# Visualise HDR (naively clipped) vs reference tone-mapped
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i in range(5):
    naive = np.clip(dataset[i]["hdr"] / dataset[i]["hdr"].max(), 0, 1)
    axes[0, i].imshow(naive)
    dr = dataset[i]["hdr"].max() / max(dataset[i]["hdr"].min(), 1e-6)
    axes[0, i].set_title(f"HDR (clipped)\nDR={dr:.0f}x", fontsize=9)
    axes[0, i].axis("off")
    axes[1, i].imshow(dataset[i]["reference"])
    axes[1, i].set_title("Reference LDR", fontsize=9)
    axes[1, i].axis("off")
plt.suptitle("Naively Clipped HDR (top) vs Reference Tone Mapped (bottom)", fontsize=13)
plt.tight_layout()
plt.show()

## 4 — Tone Mapping Operators & Quality Metrics

In [ ]:
def luminance(img: np.ndarray) -> np.ndarray:
    return 0.2126 * img[:,:,0] + 0.7152 * img[:,:,1] + 0.0722 * img[:,:,2]


def tmo_reinhard(hdr: np.ndarray, key: float = 0.18) -> np.ndarray:
    """Reinhard global TMO."""
    lum = luminance(hdr)
    log_avg = np.exp(np.mean(np.log(lum + 1e-6)))
    L = key * lum / (log_avg + 1e-6)
    L_d = L / (1 + L)
    scale = L_d / (lum + 1e-6)
    return np.clip(hdr * scale[:, :, np.newaxis], 0, 1).astype(np.float32)


def tmo_reinhard_white(hdr: np.ndarray, key: float = 0.18, white_pct: float = 95) -> np.ndarray:
    """Reinhard TMO with white point."""
    lum = luminance(hdr)
    log_avg = np.exp(np.mean(np.log(lum + 1e-6)))
    L = key * lum / (log_avg + 1e-6)
    L_white = np.percentile(L, white_pct)
    L_d = L * (1 + L / (L_white**2 + 1e-6)) / (1 + L)
    scale = L_d / (lum + 1e-6)
    return np.clip(hdr * scale[:, :, np.newaxis], 0, 1).astype(np.float32)


def tmo_log(hdr: np.ndarray, mu: float = 10.0) -> np.ndarray:
    """Logarithmic TMO."""
    lum = luminance(hdr)
    L_max = lum.max() + 1e-6
    L_d = np.log(1 + mu * lum / L_max) / np.log(1 + mu)
    scale = L_d / (lum + 1e-6)
    return np.clip(hdr * scale[:, :, np.newaxis], 0, 1).astype(np.float32)


def tmo_gamma(hdr: np.ndarray, gamma: float = 2.2) -> np.ndarray:
    """Simple gamma compression."""
    normalised = hdr / (hdr.max() + 1e-6)
    return np.clip(np.power(normalised, 1.0/gamma), 0, 1).astype(np.float32)


def tmo_local_reinhard(hdr: np.ndarray, key: float = 0.18, sigma: int = 15) -> np.ndarray:
    """Local Reinhard using Gaussian-filtered adaptation luminance."""
    lum = luminance(hdr)
    log_avg = np.exp(np.mean(np.log(lum + 1e-6)))
    L = key * lum / (log_avg + 1e-6)
    V = cv2.GaussianBlur(L, (sigma, sigma), sigma/3)
    L_d = L / (1 + V)
    scale = L_d / (lum + 1e-6)
    return np.clip(hdr * scale[:, :, np.newaxis], 0, 1).astype(np.float32)


def apply_gamma_post(img: np.ndarray, gamma: float = 2.2) -> np.ndarray:
    """Apply gamma correction as post-processing."""
    return np.clip(np.power(img, 1.0/gamma), 0, 1).astype(np.float32)


def adjust_contrast_post(img: np.ndarray, factor: float = 1.2) -> np.ndarray:
    mean = img.mean()
    return np.clip(mean + factor * (img - mean), 0, 1).astype(np.float32)

In [ ]:
def structural_fidelity(hdr: np.ndarray, ldr: np.ndarray) -> float:
    """Simplified structural fidelity component of TMQI."""
    hdr_gray = luminance(hdr)
    hdr_norm = (hdr_gray - hdr_gray.min()) / (hdr_gray.max() - hdr_gray.min() + 1e-8)
    ldr_gray = np.mean(ldr, axis=2)

    C = 0.01
    mu_x = cv2.GaussianBlur(hdr_norm, (11, 11), 1.5)
    mu_y = cv2.GaussianBlur(ldr_gray, (11, 11), 1.5)
    sigma_x = cv2.GaussianBlur(hdr_norm**2, (11, 11), 1.5) - mu_x**2
    sigma_y = cv2.GaussianBlur(ldr_gray**2, (11, 11), 1.5) - mu_y**2
    cv2.GaussianBlur(hdr_norm * ldr_gray, (11, 11), 1.5) - mu_x * mu_y

    sigma_x = np.maximum(sigma_x, 0)
    sigma_y = np.maximum(sigma_y, 0)

    S = (2 * np.sqrt(sigma_x * sigma_y) + C) / (sigma_x + sigma_y + C)
    return float(np.mean(S))


def statistical_naturalness(ldr: np.ndarray) -> float:
    """Statistical naturalness component of TMQI."""
    gray = (np.mean(ldr, axis=2) * 255).astype(np.float64)
    mu = np.mean(gray)
    sigma = np.std(gray)
    mu_n, sigma_mu_n = 115.0, 30.0
    sigma_n, sigma_sigma_n = 44.0, 15.0
    N = (np.exp(-(mu - mu_n)**2 / (2 * sigma_mu_n**2)) *
         np.exp(-(sigma - sigma_n)**2 / (2 * sigma_sigma_n**2)))
    return float(N)


def compute_tmqi(hdr: np.ndarray, ldr: np.ndarray, a=0.5, alpha=1.0, beta=1.0) -> float:
    """Tone Mapped Quality Index."""
    S = structural_fidelity(hdr, ldr)
    N = statistical_naturalness(ldr)
    return float(a * S**alpha + (1 - a) * N**beta)


# Demo: compare TMOs on one HDR image
sample_hdr = dataset[0]["hdr"]
tmos = [
    ("Reinhard",       tmo_reinhard(sample_hdr)),
    ("Reinhard+White", tmo_reinhard_white(sample_hdr)),
    ("Log (μ=10)",     tmo_log(sample_hdr, 10)),
    ("Gamma (γ=2.2)",  tmo_gamma(sample_hdr, 2.2)),
    ("Local Reinhard", tmo_local_reinhard(sample_hdr)),
    ("Reference",      dataset[0]["reference"]),
]

fig, axes = plt.subplots(1, 6, figsize=(20, 3.5))
for i, (name, img) in enumerate(tmos):
    axes[i].imshow(np.clip(img, 0, 1))
    tmqi = compute_tmqi(sample_hdr, img)
    axes[i].set_title(f"{name}\nTMQI={tmqi:.3f}", fontsize=9)
    axes[i].axis("off")
plt.suptitle("Tone Mapping Operator Comparison", fontsize=13)
plt.tight_layout()
plt.show()

## 5 — HDR Tone Mapping Environment

In [ ]:
class HDRToneMappingEnv(gym.Env):
    """RL environment for HDR tone mapping."""

    metadata = {"render_modes": ["rgb_array"]}

    ACTIONS = [
        ("reinhard_low",    lambda h: tmo_reinhard(h, key=0.09)),
        ("reinhard_mid",    lambda h: tmo_reinhard(h, key=0.18)),
        ("reinhard_high",   lambda h: tmo_reinhard(h, key=0.36)),
        ("reinhard_wp_90",  lambda h: tmo_reinhard_white(h, key=0.18, white_pct=90)),
        ("reinhard_wp_99",  lambda h: tmo_reinhard_white(h, key=0.18, white_pct=99)),
        ("log_mu5",         lambda h: tmo_log(h, mu=5)),
        ("log_mu20",        lambda h: tmo_log(h, mu=20)),
        ("log_mu50",        lambda h: tmo_log(h, mu=50)),
        ("gamma_1.8",       lambda h: tmo_gamma(h, gamma=1.8)),
        ("gamma_2.2",       lambda h: tmo_gamma(h, gamma=2.2)),
        ("gamma_2.6",       lambda h: tmo_gamma(h, gamma=2.6)),
        ("local_s7",        lambda h: tmo_local_reinhard(h, sigma=7)),
        ("local_s15",       lambda h: tmo_local_reinhard(h, sigma=15)),
        ("post_gamma_up",   None),  # post-processing
        ("post_contrast",   None),  # post-processing
        ("noop",            None),
    ]

    def __init__(self, dataset, max_steps=8):
        super().__init__()
        self.dataset = dataset
        self.max_steps = max_steps

        # State: HDR histogram features + current LDR stats
        self.feat_dim = 20
        self.observation_space = spaces.Box(-100, 100, shape=(self.feat_dim,), dtype=np.float32)
        self.action_space = spaces.Discrete(len(self.ACTIONS))

    def _extract_features(self) -> np.ndarray:
        """Extract state features from HDR image and current LDR."""
        hdr_lum = luminance(self.hdr_img)
        ldr_gray = np.mean(self.current_ldr, axis=2)

        # HDR features
        log_lum = np.log(hdr_lum + 1e-6)
        hdr_feats = [
            np.mean(log_lum), np.std(log_lum),
            np.percentile(log_lum, 5), np.percentile(log_lum, 95),
            np.log(hdr_lum.max() / (hdr_lum.min() + 1e-6)),  # dynamic range
        ]

        # LDR features
        ldr_feats = [
            np.mean(ldr_gray), np.std(ldr_gray),
            np.percentile(ldr_gray, 5), np.percentile(ldr_gray, 95),
            np.mean(ldr_gray > 0.95),  # blown-out fraction
            np.mean(ldr_gray < 0.05),  # crushed blacks fraction
        ]

        # Histogram features (5-bin histogram of LDR)
        hist, _ = np.histogram(ldr_gray, bins=5, range=(0, 1))
        hist = hist.astype(np.float32) / hist.sum()

        # Per-channel means
        ch_means = [np.mean(self.current_ldr[:,:,c]) for c in range(3)]

        # Step fraction
        step_frac = [self.step_count / self.max_steps]

        features = np.array(hdr_feats + ldr_feats + hist.tolist() + ch_means + step_frac, dtype=np.float32)
        return features

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        idx = random.randint(0, len(self.dataset) - 1)
        self.hdr_img = self.dataset[idx]["hdr"].copy()
        self.ref_ldr = self.dataset[idx]["reference"].copy()
        # Start with a naive clip as initial LDR
        self.current_ldr = np.clip(self.hdr_img / (self.hdr_img.max() + 1e-6), 0, 1).astype(np.float32)
        self.step_count = 0
        self.prev_tmqi = compute_tmqi(self.hdr_img, self.current_ldr)
        return self._extract_features(), {"tmqi": self.prev_tmqi}

    def step(self, action: int):
        name, fn = self.ACTIONS[action]

        if name == "post_gamma_up":
            self.current_ldr = apply_gamma_post(self.current_ldr, 1.5)
        elif name == "post_contrast":
            self.current_ldr = adjust_contrast_post(self.current_ldr, 1.3)
        elif name == "noop":
            pass
        else:
            self.current_ldr = fn(self.hdr_img)

        self.step_count += 1
        tmqi = compute_tmqi(self.hdr_img, self.current_ldr)
        ssim = float(compute_ssim(self.ref_ldr, self.current_ldr, data_range=1.0, channel_axis=2))

        reward = (tmqi - self.prev_tmqi) + 0.5 * ssim
        self.prev_tmqi = tmqi

        terminated = self.step_count >= self.max_steps
        return self._extract_features(), float(reward), terminated, False, {"tmqi": tmqi, "ssim": ssim}


env = HDRToneMappingEnv(dataset)
obs, info = env.reset()
print(f"Feature dim: {obs.shape[0]}, Actions: {env.action_space.n}")
print(f"Initial TMQI: {info['tmqi']:.4f}")

## 6 — DQN Agent & Training

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, feat_dim, num_actions):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feat_dim, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, num_actions),
        )

    def forward(self, x):
        return self.net(x)


class ReplayBuffer:
    def __init__(self, capacity):
        self.buf = collections.deque(maxlen=capacity)

    def push(self, *t):
        self.buf.append(t)

    def sample(self, bs):
        batch = random.sample(self.buf, bs)
        s, a, r, ns, d = zip(*batch)
        return (torch.FloatTensor(np.array(s)).to(device),
                torch.LongTensor(a).to(device),
                torch.FloatTensor(r).to(device),
                torch.FloatTensor(np.array(ns)).to(device),
                torch.FloatTensor(d).to(device))

    def __len__(self):
        return len(self.buf)

In [ ]:
NUM_EPISODES = 500
BATCH_SIZE = 64
GAMMA = 0.99
LR = 5e-4
BUFFER_SIZE = 15_000
EPS_START, EPS_END, EPS_DECAY = 1.0, 0.05, 350
TARGET_UPDATE = 10
MAX_STEPS = 8

policy_net = QNetwork(env.feat_dim, env.action_space.n).to(device)
target_net = QNetwork(env.feat_dim, env.action_space.n).to(device)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

optimizer = optim.Adam(policy_net.parameters(), lr=LR)
replay_buffer = ReplayBuffer(BUFFER_SIZE)

ep_rewards, ep_tmqis, ep_ssims, losses = [], [], [], []
global_step = 0

for ep in range(NUM_EPISODES):
    obs, info = env.reset()
    ep_reward = 0.0

    for t in range(MAX_STEPS):
        eps = EPS_END + (EPS_START - EPS_END) * math.exp(-global_step / EPS_DECAY)
        if random.random() < eps:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                action = policy_net(torch.FloatTensor(obs).unsqueeze(0).to(device)).argmax(1).item()

        next_obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        replay_buffer.push(obs, action, reward, next_obs, float(done))
        obs = next_obs
        ep_reward += reward
        global_step += 1

        if len(replay_buffer) >= BATCH_SIZE:
            states, actions, rewards, next_states, dones = replay_buffer.sample(BATCH_SIZE)
            q_vals = policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                next_q = target_net(next_states).max(1)[0]
                target_q = rewards + GAMMA * next_q * (1 - dones)
            loss = F.smooth_l1_loss(q_vals, target_q)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(policy_net.parameters(), 1.0)
            optimizer.step()
            losses.append(loss.item())

        if done:
            break

    ep_rewards.append(ep_reward)
    ep_tmqis.append(info["tmqi"])
    ep_ssims.append(info["ssim"])

    if ep % TARGET_UPDATE == 0:
        target_net.load_state_dict(policy_net.state_dict())

    if ep % 100 == 0:
        print(f"Ep {ep:4d} | Reward: {np.mean(ep_rewards[-50:]):7.3f} | TMQI: {np.mean(ep_tmqis[-50:]):.4f} | SSIM: {np.mean(ep_ssims[-50:]):.4f} | ε: {eps:.3f}")

print("\nTraining complete!")

## 7 — Training Curves

In [ ]:
def moving_avg(d, w=30):
    return np.convolve(d, np.ones(w)/w, mode="valid")

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

axes[0].plot(ep_rewards, alpha=0.25, color="steelblue")
axes[0].plot(moving_avg(ep_rewards), color="navy", lw=2)
axes[0].set(xlabel="Episode", ylabel="Reward", title="Episode Reward")
axes[0].grid(True, alpha=0.3)

axes[1].plot(ep_tmqis, alpha=0.25, color="coral")
axes[1].plot(moving_avg(ep_tmqis), color="darkred", lw=2)
axes[1].set(xlabel="Episode", ylabel="TMQI", title="Tone Map Quality Index")
axes[1].grid(True, alpha=0.3)

axes[2].plot(ep_ssims, alpha=0.25, color="orchid")
axes[2].plot(moving_avg(ep_ssims), color="purple", lw=2)
axes[2].set(xlabel="Episode", ylabel="SSIM", title="SSIM vs Reference")
axes[2].grid(True, alpha=0.3)

axes[3].plot(losses[:1000], alpha=0.25, color="mediumseagreen")
if len(losses) > 30:
    axes[3].plot(moving_avg(losses[:1000]), color="darkgreen", lw=2)
axes[3].set(xlabel="Step", ylabel="Loss", title="DQN Loss")
axes[3].grid(True, alpha=0.3)

plt.suptitle("HDR Tone Mapping Agent — Training Dynamics", fontsize=14)
plt.tight_layout()
plt.show()

## 8 — Evaluation: Before / After

In [ ]:
def evaluate_hdr_agent(env, policy_net, n=5):
    results = []
    policy_net.eval()
    for _ in range(n):
        obs, info = env.reset()
        init_tmqi = info["tmqi"]
        naive = env.current_ldr.copy()
        actions_seq = []
        for t in range(MAX_STEPS):
            with torch.no_grad():
                action = policy_net(torch.FloatTensor(obs).unsqueeze(0).to(device)).argmax(1).item()
            actions_seq.append(action)
            obs, _, done, _, info = env.step(action)
            if done:
                break
        results.append({
            "hdr": env.hdr_img.copy(),
            "naive": naive,
            "agent_ldr": env.current_ldr.copy(),
            "reference": env.ref_ldr.copy(),
            "tmqi_before": init_tmqi,
            "tmqi_after": info["tmqi"],
            "ssim": info["ssim"],
            "actions": actions_seq,
        })
    policy_net.train()
    return results

results = evaluate_hdr_agent(env, policy_net)

fig, axes = plt.subplots(4, 5, figsize=(20, 16))
row_labels = ["HDR (clipped)", "Naive LDR", "Agent Output", "Reference"]
for i, r in enumerate(results):
    axes[0, i].imshow(np.clip(r["hdr"] / (r["hdr"].max() + 1e-6), 0, 1))
    axes[0, i].set_title("HDR (clipped)", fontsize=9); axes[0, i].axis("off")

    axes[1, i].imshow(np.clip(r["naive"], 0, 1))
    axes[1, i].set_title(f"Naive\nTMQI={r['tmqi_before']:.3f}", fontsize=9); axes[1, i].axis("off")

    axes[2, i].imshow(np.clip(r["agent_ldr"], 0, 1))
    axes[2, i].set_title(f"Agent\nTMQI={r['tmqi_after']:.3f}", fontsize=9); axes[2, i].axis("off")

    axes[3, i].imshow(r["reference"])
    ref_tmqi = compute_tmqi(r["hdr"], r["reference"])
    axes[3, i].set_title(f"Reference\nTMQI={ref_tmqi:.3f}", fontsize=9); axes[3, i].axis("off")

for row, label in enumerate(row_labels):
    axes[row, 0].set_ylabel(label, fontsize=11)

plt.suptitle("HDR Tone Mapping: Naive → Agent → Reference", fontsize=14)
plt.tight_layout()
plt.show()

print(f"{'Img':>4} | {'TMQI Before':>11} | {'TMQI After':>10} | {'SSIM':>6} | {'Selected TMO (final)':>25}")
print("-" * 70)
action_names = [name for name, _ in HDRToneMappingEnv.ACTIONS]
for i, r in enumerate(results):
    last_action = action_names[r['actions'][-1]] if r['actions'] else 'none'
    print(f"{i:4d} | {r['tmqi_before']:11.4f} | {r['tmqi_after']:10.4f} | {r['ssim']:6.4f} | {last_action:>25}")

## 9 — TMO Selection Analysis

In [ ]:
# Analyse which TMOs the agent prefers
action_names = [name for name, _ in HDRToneMappingEnv.ACTIONS]
all_actions = []
for _ in range(200):
    obs, _ = env.reset()
    policy_net.eval()
    for t in range(MAX_STEPS):
        with torch.no_grad():
            action = policy_net(torch.FloatTensor(obs).unsqueeze(0).to(device)).argmax(1).item()
        all_actions.append(action)
        obs, _, done, _, _ = env.step(action)
        if done:
            break

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

counts = np.bincount(all_actions, minlength=len(action_names))
colours = plt.cm.Set3(np.linspace(0, 1, len(action_names)))
ax1.barh(action_names, counts, color=colours)
ax1.set_xlabel("Frequency (200 episodes)")
ax1.set_title("TMO Action Distribution")
ax1.grid(True, alpha=0.3, axis="x")

# Show dynamic range vs preferred TMO
dr_values = []
chosen_actions = []
policy_net.eval()
for d in dataset[:30]:
    hdr = d["hdr"]
    lum = luminance(hdr)
    dr = np.log10(lum.max() / (lum.min() + 1e-6))
    naive = np.clip(hdr / (hdr.max() + 1e-6), 0, 1).astype(np.float32)

    env_temp = HDRToneMappingEnv(dataset)
    obs, _ = env_temp.reset()
    env_temp.hdr_img = hdr
    env_temp.current_ldr = naive
    obs = env_temp._extract_features()

    with torch.no_grad():
        action = policy_net(torch.FloatTensor(obs).unsqueeze(0).to(device)).argmax(1).item()
    dr_values.append(dr)
    chosen_actions.append(action)

# Group by TMO type
global_mask = [a < 8 for a in chosen_actions]
gamma_mask = [8 <= a <= 10 for a in chosen_actions]
local_mask = [11 <= a <= 12 for a in chosen_actions]

for mask, label, color in [(global_mask, "Global TMO", "royalblue"),
                            (gamma_mask, "Gamma", "coral"),
                            (local_mask, "Local TMO", "forestgreen")]:
    drs = [dr_values[i] for i, m in enumerate(mask) if m]
    if drs:
        ax2.scatter(drs, [1]*len(drs), label=label, alpha=0.6, s=100, color=color)

ax2.set_xlabel("Dynamic Range (log₁₀)")
ax2.set_title("First Action vs Image Dynamic Range")
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_yticks([])

plt.suptitle("Learned Tone Mapping Strategy", fontsize=14)
plt.tight_layout()
plt.show()

## 10 — Luminance Histogram Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (ax, r) in enumerate(zip(axes, results[:3])):
    lum_naive = np.mean(r["naive"], axis=2).ravel()
    lum_agent = np.mean(r["agent_ldr"], axis=2).ravel()
    lum_ref = np.mean(r["reference"], axis=2).ravel()

    ax.hist(lum_naive, bins=50, alpha=0.4, color="gray", label="Naive", density=True)
    ax.hist(lum_agent, bins=50, alpha=0.5, color="steelblue", label="Agent", density=True)
    ax.hist(lum_ref, bins=50, alpha=0.4, color="coral", label="Reference", density=True)
    ax.set_xlabel("Pixel Intensity")
    ax.set_ylabel("Density")
    ax.set_title(f"Image {i}")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle("Luminance Histograms: Naive vs Agent vs Reference", fontsize=14)
plt.tight_layout()
plt.show()

## 11 — Key Takeaways

| Insight | Detail |
|---------|--------|
| **TMO selection** | Different HDR images benefit from different tone mapping strategies |
| **TMQI metric** | Combines structural fidelity and statistical naturalness for perceptually meaningful evaluation |
| **Dynamic range adaptation** | High-DR scenes often need local operators; low-DR can use simpler gamma correction |
| **Sequential refinement** | The agent can apply a base TMO then fine-tune with post-processing actions |
| **Extensions** | Continuous parameter tuning via actor-critic, real HDR datasets, spatially-varying local TMOs |

---

*End of Module 7: RL for Image Enhancement. These five notebooks demonstrated how reinforcement learning agents can learn to make intelligent image processing decisions across brightness/contrast adjustment, colour correction, denoising, super-resolution, and HDR tone mapping.*